# Tutorial 6: Agentic RAG, Letting the Model Drive Retrieval

Tutorial 3, section 7 built RAG as a chain: retrieve once, then answer. The path is fixed, so every question gets
exactly one search with the user's own words, and whatever comes back goes into the prompt. That is fine for simple
lookups and breaks in predictable ways on everything else.

**Agentic RAG** moves the retrieval decisions out of the fixed pipe and into the control flow you learned in tutorials
4 and 5: whether to search at all, what to search for, whether the results are any good, and whether to search again.

By the end you will be able to:

- name the four ways a **retrieve-once chain** fails, and recognise each one in a trace;
- turn a retriever into a **tool** and let a `create_agent` agent decide when and how often to search;
- build a **corrective RAG graph** by hand: route, retrieve, grade, rewrite, generate, check;
- explain what **Self-RAG** style checks cost and what they buy;
- handle **multi-hop** questions by wrapping the graph as a tool for a planning agent;
- **measure** accuracy and token cost instead of judging a RAG system from one example;
- choose between a chain, an agent and a graph for a retrieval problem, with a reason.

## 0. Mental model: from a pipe to a loop

<img src="images/agentic_rag_cards/00_pipe_to_loop.png" alt="Agentic RAG card: on the left, a ride that searches the library once even for &#x27;Thanks!&#x27;; on the right, a conductor at a lever chooses between a shortcut exit and a loop back to the library. Search once versus choose or loop." width="650">

Tutorial 3's chain, and where it goes wrong:

```text
question → retrieve(question) → prompt(context, question) → answer
              │                          │
              │                          └─ uses whatever came back, relevant or not
              └─ always runs, once, with the user's exact words
```

| Failure | Example | What would fix it |
|---|---|---|
| **No retrieval was needed** | "Thanks, that helped!" | Decide *whether* to retrieve |
| **The wrong chunks came back** | A late-policy question retrieves another course's late policy | *Grade* the results before using them |
| **The query was badly worded** | "the team thing deadline?" | *Rewrite* the query and search again |
| **One search cannot find it** | "Where are the office hours of whoever teaches the KV cache session?" | Search *several times*, each search using the last result |

Every fix is a decision, and tutorial 4 showed where decisions live: in an agent's loop or in a graph's conditional
edges. That gives two families of agentic RAG, and this notebook builds both:

```text
 agent:  question → [model ⇄ search tool]* → answer           the model decides everything
 graph:  question → route → retrieve → grade ─→ generate → check → answer
                                 ▲       │
                                 └ rewrite                     you decide the steps, the model fills them in
```

The rule from tutorial 4 still applies: **a chain if the path is known, a graph if a decision is needed**. Most RAG
questions need one or two decisions, not a fully autonomous agent.

## 1. Setup

<img src="images/agentic_rag_cards/01_setup.png" alt="The robot&#x27;s RAG workbench card: a handbook goes through local embeddings into an in-memory index on the desk, while a remote chat model in a cloud needs a key." width="650">

Same environment as tutorials 3 and 4:

```bash
pip install "langchain>=1.0" langchain-core langchain-groq langgraph python-dotenv
pip install langchain-huggingface sentence-transformers
```

and the same `.env` beside this notebook:

```text
GROQ_API_KEY=your-key-here
```

Agentic RAG makes many small model calls per question. If you hit Groq's rate limits, wait a minute and re-run the
cell, or switch the model below.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY not found, see tutorial 3, section 1."

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

model = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

LIMITS = {"recursion_limit": 25}


def show_trace(messages, width=160):
    """Print one line per message: who spoke, and either the tool calls or the text."""
    for m in messages:
        kind = getattr(m, "name", None) or type(m).__name__
        calls = getattr(m, "tool_calls", None)
        if calls:
            body = "→ " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        else:
            body = str(m.content).replace("\n", " ")
        print(f"{kind:>14}: {body[:width]}")


print("model and embeddings ready")

### Code walkthrough: what is new

<img src="images/agentic_rag_cards/01a_setup_walkthrough.png" alt="Read the tool trace card: a vague question, &#x27;The big assignment?&#x27;, and a search trace in which a magnifying glass shows the query the model actually wrote: &#x27;LLM-501 final project&#x27;." width="650">

**`InMemoryVectorStore`** replaces tutorial 3's Chroma. It has the same interface (`from_documents`,
`as_retriever`, `similarity_search`) and needs no extra package. Chroma keeps one shared collection per process, so
building a second store in the same notebook silently adds to the first; the in-memory store does not have that trap.

**`HuggingFaceEmbeddings`** is the same local `all-MiniLM-L6-v2` model as tutorial 3. Only the chat model is remote.

**`show_trace`** is tutorial 5's helper: one line per message, with the arguments of each tool call. In agentic RAG the
arguments are the interesting part, because they are the search queries the model chose.

## 2. A handbook that breaks naive RAG

<img src="images/agentic_rag_cards/02_handbook.png" alt="The handbook card: a puzzled robot asks for the LLM-501 late penalty while a claw machine grabs two similar tags, LLM-501 10% per day and NLP-402 20% per day." width="650">

The corpus is a small, made-up course handbook for two courses. It is made up on purpose: the model cannot know any of
it from training, so every correct answer must have come from retrieval. It is also built to cause the four failures
from section 0: the two courses have similar rules with different numbers, and several facts are spread over two
documents.

In [ ]:
HANDBOOK = {
    "llm-syllabus": "LLM-501 (Large Language Models) is taught by Dr. Navid Rahimi. Grading: final project 40%, "
                    "final exam 35%, midterm 25%.",
    "llm-schedule": "In LLM-501, the week 9 session on the KV cache and fast decoding is led by the teaching "
                    "assistant Sara Karimi. The week 10 session on retrieval-augmented generation is led by "
                    "Dr. Navid Rahimi.",
    "office-hours": "Sara Karimi holds office hours on Tuesdays from 14:00 to 16:00 in room 305. Ali Moradi holds "
                    "office hours on Sundays from 10:00 to 12:00 in room 112.",
    "llm-project": "The LLM-501 final project is done in teams of two and must be built with LangGraph or Deep "
                   "Agents. Proposals are due in week 8, and the final report is due one week after the last session.",
    "llm-late-policy": "LLM-501 late policy: a late submission loses 10% of its grade per day, for up to three days. "
                       "After three days it receives zero.",
    "nlp-syllabus": "NLP-402 (Natural Language Processing) is taught by Dr. Leila Hosseini. Grading: homework 30%, "
                    "midterm 30%, final exam 40%.",
    "nlp-late-policy": "NLP-402 late policy: late homework loses 20% of its grade per day and is not accepted after "
                       "two days.",
    "nlp-schedule": "In NLP-402, the sessions on word embeddings (week 4) and on attention (week 7) are led by the "
                    "teaching assistant Ali Moradi.",
    "prerequisites": "LLM-501 requires NLP-402 or an equivalent course. Students without it need written permission "
                     "from the instructor of LLM-501.",
    "exams": "The LLM-501 final exam is on 2026-12-20 in hall A. The NLP-402 final exam is on 2026-12-27 in hall B. "
             "A student who misses a final exam for medical reasons may take a make-up exam within two weeks, with "
             "a medical certificate.",
    "compute": "Each LLM-501 project team gets 20 GPU hours on the department cluster. Requests for more hours go to "
               "the cluster manager, Ali Moradi.",
    "nlp-lab": "Room 112 is the NLP lab. It is directed by Dr. Leila Hosseini, and Ali Moradi is its lab engineer.",
}

docs = [Document(page_content=text, metadata={"source": src}) for src, text in HANDBOOK.items()]
store = InMemoryVectorStore.from_documents(docs, embeddings)
retriever = store.as_retriever(search_kwargs={"k": 2})


def format_docs(docs):
    return "\n\n".join(f"[{d.metadata['source']}] {d.page_content}" for d in docs)


print(len(docs), "documents indexed")

### Code walkthrough: no splitter this time

<img src="images/agentic_rag_cards/02a_no_splitter.png" alt="One entry, one chunk card: the scissors are put away, each whole handbook entry is filed in the index, and the search returns the top two cards." width="650">

Each entry is already one or two sentences, so there is nothing to split; each `Document` is one chunk. With real files
you would put tutorial 3's `RecursiveCharacterTextSplitter` between loading and indexing, and nothing below would
change.

**`k=2`** matches tutorial 3. It keeps the context short, and it also means a question that needs three documents
cannot be answered by one search. Keep that in mind when you read the baseline below.

### The baseline: tutorial 3's chain on harder questions

<img src="images/agentic_rag_cards/02b_baseline.png" alt="The retrieve-once baseline card: a conveyor belt runs from &#x27;use the question&#x27; to &#x27;search once&#x27; (chunk 1 and chunk 2) to &#x27;answer with both chunks&#x27;." width="650">

Here is tutorial 3's chain again, written as a function that also returns the documents it used, so you can see *why*
an answer is right or wrong.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the question using only the context below, and cite the [source] tags you used. "
     "If the context does not contain the answer, say so. Do not guess.\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])
answer_chain = rag_prompt | model | StrOutputParser()


def naive_rag(question):
    found = retriever.invoke(question)
    answer = answer_chain.invoke({"context": format_docs(found), "question": question})
    return answer, found


QUESTIONS = {
    "chit-chat":   "Thanks, that was really helpful!",
    "distractor":  "How much do I lose per day if I hand in my LLM project late?",
    "vague":       "is it ok to work alone on the big assignment?",
    "multi-hop":   "I missed the KV cache lecture. Where can I find the person who gave it?",
    "not covered": "Is there free parking for students?",
}

for label, question in QUESTIONS.items():
    answer, found = naive_rag(question)
    print(f"[{label}] {question}")
    print("   retrieved:", [d.metadata["source"] for d in found])
    print("   answer:   ", answer.replace("\n", " ")[:300])
    print()

### Code walkthrough: reading the failures

<img src="images/agentic_rag_cards/02c_failures.png" alt="Four ways retrieval can go wrong card: an unneeded search for &#x27;Thanks!&#x27;, a wrong document (a late policy stamped wrong course), a vague query that misses its target, and a missing hop where a bridge between the search and the answer is broken." width="650">

Your exact answers will differ from run to run and model to model, so read the `retrieved:` line first. It tells you
whether the answer *could* have been right.

- **chit-chat.** The chain searched the handbook for "thanks", got two unrelated documents, and the grounding
  instruction probably made the model say the context does not cover it. Nothing was wrong with retrieval; it should
  never have run.
- **distractor.** If `nlp-late-policy` came back next to (or instead of) `llm-late-policy`, the model has two
  numbers to choose from. Sometimes it picks the right one. "Sometimes" is the problem.
- **vague.** "work alone on the big assignment" shares no words with "the final project is done in teams of two",
  and `llm-project` is not among the retrieved documents. No prompt can save an answer whose evidence never arrived.
- **multi-hop.** The answer needs `llm-schedule` (who gave the lecture) *and* `office-hours` (where that person can
  be found). The question mentions neither "Sara" nor "office hours", so the search finds the schedule, which matches
  "KV cache lecture", plus one unrelated document. The second fact can only be searched for once the first is known.
- **not covered.** This is the one the chain handles well, thanks to the prompt. Keep it in the test set anyway: a
  more "agentic" system can lose this property, because a model that is allowed to search repeatedly is also
  tempted to keep going until it finds something it can stretch into an answer.

The pattern from tutorial 3 holds: **what breaks is retrieval, not generation.** Every fix in this notebook is about
what reaches the prompt.

## 3. The simplest agentic RAG: retrieval as a tool

<img src="images/agentic_rag_cards/03_tool.png" alt="Retrieval as a tool card: for &#x27;Thanks!&#x27; the librarian robot replies directly; for &#x27;When is the exam?&#x27; it calls search_handbook and looks up the exam card." width="650">

Tutorial 4 gave a model tools and a loop. A retriever is just another tool. Wrap it with `@tool`, hand it to
`create_agent`, and the model decides whether to search, what to search for, and how many times.

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def search_handbook(query: str) -> str:
    """Search the course handbook for LLM-501 and NLP-402: staff, sessions, grading, deadlines,
    late policies, exams, rooms, office hours and compute. Use a short query about one thing.
    If the results mention a person, room or course you need to know more about, search again for it."""
    return format_docs(retriever.invoke(query))


rag_agent = create_agent(
    model,
    tools=[search_handbook],
    system_prompt=(
        "You answer students' questions about two courses, LLM-501 and NLP-402. "
        "For any question about the courses, use search_handbook and answer only from what it returns, "
        "citing the [source] tags. A question may need several searches. "
        "For greetings and thanks, reply briefly without searching. "
        "If the handbook does not cover the question, say so."
    ),
    name="rag_agent",
)


def ask(agent, prompt):
    return agent.invoke({"messages": [HumanMessage(prompt)]}, config=LIMITS)


for label in ["chit-chat", "vague", "multi-hop"]:
    print(f"===== {label}: {QUESTIONS[label]}")
    show_trace(ask(rag_agent, QUESTIONS[label])["messages"])
    print()

### Code walkthrough: three decisions moved into the model

<img src="images/agentic_rag_cards/03a_decisions.png" alt="Three choices at the search controls card: a robot drives with three controls, whether to search, what to search for (a written query) and how often to search." width="650">

Read the `search_handbook(...)` lines in the traces. They are the queries the *model* wrote, not the user:

| Decision | Chain (section 2) | Agent |
|---|---|---|
| Search at all? | Always | The model decides; "thanks" usually gets no search |
| What to search for? | The user's exact words | The model's own query, often clearer than the user's |
| How many searches? | Exactly one | As many as it needs, up to `recursion_limit` |

**Multi-hop now works the natural way**: the first search finds that Sara Karimi gave the lecture, and the model
writes a second query about Sara Karimi's office hours. That second query could not have been written before the
first result came back, which is exactly why a fixed pipe cannot do it.

**The docstring is the retrieval policy.** "Use a short query about one thing" and "search again" are instructions
to the model about *how* to retrieve. Change them and the behaviour changes.

What you gave up: every decision is now the model's. Nothing checks that the retrieved text is relevant, nothing stops
the model from answering from its own weights, and a weak model may stop after one search. The next section takes the
decisions back, one by one, into a graph you control.

### Exercise 3.1

<img src="images/agentic_rag_cards/03b_exercise.png" alt="Change the instructions card: a robot edits the &#x27;search again&#x27; line of the tool docstring, compares two traces and presses Test. Edit, rerun, count searches." width="650">

1. Delete the last sentence of the `search_handbook` docstring ("If the results mention...") and re-run the
   multi-hop question. Does the agent still do a second search?
2. Ask the agent `QUESTIONS["not covered"]`. Count its searches. Does it admit the handbook has no answer, or does it
   stretch an unrelated document into one?

## 4. Corrective RAG: a graph you control

<img src="images/agentic_rag_cards/04_corrective.png" alt="Corrective RAG card: an inspector scans document cards on a conveyor, keeps the relevant card for the writer and throws the irrelevant one in a bin; if none are kept, rewrite the query and search again." width="650">

This section builds the graph from section 0 by hand. Each node is a small, single-purpose step, and every place where
the agent made an invisible decision becomes a node you can inspect:

```text
START → route ──direct──→ direct_answer → END
          │
       handbook
          ▼
       retrieve ◄──────────── rewrite
          ▼                      ▲
        grade ──none, rewrites left┘
          │
          ├──some relevant──────────→ generate → END
          └──none, no rewrites left─→ give_up  → END
```

The design is known in the literature as **Corrective RAG (CRAG)**: grade what came back, and correct course before
generating when it is not good enough.

### 4.1 Three small judges

<img src="images/agentic_rag_cards/04a_judges.png" alt="Three small judges card: one judge routes the message to Direct or Handbook, one grades a document with a tick, and one checks the answer against the source." width="650">

Three of the decisions are classifications, so they use **structured output**: the model must return a validated
object, and the graph's edges read a field instead of searching free text.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field


class Route(BaseModel):
    """Where a student's message should go."""
    destination: Literal["handbook", "direct"] = Field(
        description="'handbook' for anything about the courses, their staff, rules, dates, rooms or resources; "
                    "'direct' for greetings, thanks and small talk."
    )


class Grade(BaseModel):
    """Whether one retrieved document is useful for the question."""
    relevant: bool = Field(description="True if the document contains information that helps answer the question.")


router = model.with_structured_output(Route)
grader = model.with_structured_output(Grade)

print(router.invoke("Thanks, that was really helpful!"))
print(router.invoke("When is the LLM-501 final exam?"))
print(grader.invoke(
    "Question: How much do I lose per day for a late LLM-501 project?\n\n"
    f"Document: {HANDBOOK['nlp-late-policy']}"
))

### Code walkthrough: classifiers, not chat

<img src="images/agentic_rag_cards/04a_structured.png" alt="Fixed verdicts drive the graph card: a structured output machine prints only Direct or Handbook tickets, and the ticket switches the track to the matching station." width="650">

**`model.with_structured_output(Route)`** returns a runnable whose output is a `Route` object, not a message. Under the
hood the schema is sent as a tool definition and the reply is parsed and validated, the same mechanism as tutorial 4's
`ToolStrategy(Review)`.

**`Literal["handbook", "direct"]`** limits the answer to two labels. The router cannot invent a third one, so the
conditional edge that reads it can never receive a label it has no path for.

**The field descriptions are the prompt.** The router has no system prompt of its own; the model learns what
"handbook" means from the description. This is the same idea as "the docstring is the interface" in tutorial 3.

**The grader just rejected a document that sounds right.** The NLP late policy is about late submissions, and a
similarity search ranks it highly, but it is about the wrong course. Similarity is not relevance, and the grader is a
second, more careful opinion on the retriever's ranking.

### 4.2 The graph

<img src="images/agentic_rag_cards/04b_graph.png" alt="Keep the question, change the query card: the original question is locked under a glass dome while a robot erases and rewrites the search query, with retry tickets and a stop sign limiting the loop." width="650">

The state holds the question, the current search query (which may be rewritten), the documents that survived grading,
and counters that guarantee the loop ends.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

MAX_REWRITES = 2


class RAGState(TypedDict, total=False):
    question: str
    route: str
    query: str
    documents: list[Document]
    rewrites: int
    answer: str
    grounded: bool
    attempts: int


def route(state: RAGState) -> RAGState:
    return {"route": router.invoke(state["question"]).destination}


def direct_answer(state: RAGState) -> RAGState:
    reply = model.invoke([SystemMessage("You are a friendly course assistant. Reply in one or two sentences."),
                          HumanMessage(state["question"])])
    return {"answer": reply.content}


def retrieve(state: RAGState) -> RAGState:
    return {"documents": retriever.invoke(state["query"])}


def grade(state: RAGState) -> RAGState:
    kept = [d for d in state["documents"]
            if grader.invoke(f"Question: {state['question']}\n\nDocument: {d.page_content}").relevant]
    return {"documents": kept}


REWRITE_PROMPT = (
    "A search of a course handbook found nothing useful for the student's question.\n"
    "Question: {question}\nQuery that failed: {query}\n\n"
    "Write one new search query, short and specific, using the words a handbook would use "
    "(for example 'final project deadline' rather than 'team thing due'). Reply with the query only."
)


def rewrite(state: RAGState) -> RAGState:
    new_query = model.invoke(REWRITE_PROMPT.format(question=state["question"], query=state["query"]))
    return {"query": new_query.content.strip(), "rewrites": state["rewrites"] + 1}


def generate(state: RAGState) -> RAGState:
    question = state["question"]
    if state.get("grounded") is False:
        question += ("\n\n(Your previous answer contained claims the context does not support. "
                     "Answer again, using only facts stated in the context.)")
    answer = answer_chain.invoke({"context": format_docs(state["documents"]), "question": question})
    return {"answer": answer}


def give_up(state: RAGState) -> RAGState:
    return {"answer": "The handbook does not seem to cover this. Please ask the course staff."}


def after_route(state: RAGState) -> str:
    return state["route"]


def after_grade(state: RAGState) -> str:
    if state["documents"]:
        return "generate"
    if state["rewrites"] < MAX_REWRITES:
        return "rewrite"
    return "give_up"


def build_rag_graph(check_answers=False):
    builder = StateGraph(RAGState)
    for name, node in [("route", route), ("direct_answer", direct_answer), ("retrieve", retrieve),
                       ("grade", grade), ("rewrite", rewrite), ("generate", generate), ("give_up", give_up)]:
        builder.add_node(name, node)

    builder.add_edge(START, "route")
    builder.add_conditional_edges("route", after_route, {"direct": "direct_answer", "handbook": "retrieve"})
    builder.add_edge("retrieve", "grade")
    builder.add_conditional_edges("grade", after_grade,
                                  {"generate": "generate", "rewrite": "rewrite", "give_up": "give_up"})
    builder.add_edge("rewrite", "retrieve")
    builder.add_edge("direct_answer", END)
    builder.add_edge("give_up", END)

    if check_answers:                       # section 4.3
        builder.add_node("check", check)
        builder.add_edge("generate", "check")
        builder.add_conditional_edges("check", after_check, {"retry": "generate", "done": END})
    else:
        builder.add_edge("generate", END)
    return builder.compile()


crag = build_rag_graph()
print(crag.get_graph().draw_mermaid())

### Code walkthrough: one decision per node

<img src="images/agentic_rag_cards/04b_nodes.png" alt="One job per node card: on a conveyor, the grade robot removes a bad document and the rewrite robot rewrites the query, each touching only its own tray. Return only what changed." width="650">

**Every node does one thing and returns only the keys it changed**, as in tutorial 4. `grade` returns a shorter
`documents` list; `rewrite` returns a new `query` and a bigger counter. Nothing else is touched.

**`total=False`** on the `TypedDict` says no key is required. The graph starts with only `question`, `query` and
`rewrites`, and the other keys appear as nodes fill them in.

**`question` and `query` are kept apart on purpose.** The grader and the generator always see the student's real
question. Only the retriever sees the rewritten query, because the rewrite is for the search engine, not for the
answer.

**`MAX_REWRITES`** is tutorial 4's rule that every loop needs a limit that does not depend on the model behaving.
After two rewrites the graph gives up honestly instead of answering from nothing. `give_up` is a plain function with
no model call: a fixed message is the safest possible answer when the evidence is missing.

**`build_rag_graph`** builds two variants from one function, so section 4.3 can add one node without copying the
whole graph. `check` and `after_check` are defined there; Python only looks them up when `check_answers=True`.

Paste the printed Mermaid text into [mermaid.live](https://mermaid.live) to see the diagram.

### Watching the graph decide

<img src="images/agentic_rag_cards/04b_watching.png" alt="Watch the graph decide card: a conductor reads a printed trace, route, retrieve, grade, rewrite, retrieve, while a toy train loops back to the bookshelf." width="650">

`stream(..., stream_mode="updates")` yields one item per node, as it finishes: the node's name and the keys it
returned. Printing them shows the path each question took.

In [ ]:
def describe(update):
    parts = []
    for key, value in (update or {}).items():
        if key == "documents":
            parts.append(f"documents={[d.metadata['source'] for d in value]}")
        elif key == "answer":
            parts.append(f"answer={value.strip()[:120]!r}")
        else:
            parts.append(f"{key}={value!r}")
    return ", ".join(parts)


def run_graph(graph, question, verbose=True):
    state = {"question": question, "query": question, "rewrites": 0, "attempts": 0}
    for step in graph.stream(state, config=LIMITS, stream_mode="updates"):
        for node, update in step.items():
            state.update(update or {})
            if verbose:
                print(f"  {node:>13}: {describe(update)}")
    return state


for label, question in QUESTIONS.items():
    print(f"===== {label}: {question}")
    run_graph(crag, question)
    print()

### Code walkthrough: reading the paths

<img src="images/agentic_rag_cards/04b_paths.png" alt="Follow the route card: a conductor at a junction sends trains to four stations: direct reply, generate, rewrite and give up." width="650">

Compare each path with the section 2 baseline:

- **chit-chat** goes `route → direct_answer`. No retrieval, no confused "the context does not say".
- **distractor**: look at `grade`. If `nlp-late-policy` was retrieved, it should be gone after grading, so the
  generator only ever sees the LLM-501 rule.
- **vague**: the first search misses, so the path shows `grade: documents=[]`, then `rewrite: query='...'`, then a
  second `retrieve`. The rewritten query is worth reading: it is the model translating student language into
  handbook language.
- **not covered**: nothing survives grading, the query is rewritten twice, and the graph ends in `give_up`. That costs
  more calls than the chain, but the answer is a reliable "I don't know".
- **multi-hop** is the one this graph does *not* fix. It searches once per query, so at best it finds one of the two
  documents and answers half the question. Section 5 deals with it.

**Cost.** Grading calls the model once per retrieved document, so a question now costs `1 (route) + k (grade) +
1 (generate)` calls when all goes well, and more with rewrites. Section 6 measures it.

### 4.3 Self-checking the answer

<img src="images/agentic_rag_cards/04c_selfrag.png" alt="Self-RAG card: a robot ties each line of a draft to the source; &#x27;Room 305&#x27; matches the office hours entry, while &#x27;Free pizza&#x27; has no source and is rejected. Check the draft against the source." width="650">

Grading checks the *input* to the generator. A model can still add facts that are in none of the documents,
especially when the context is close to, but not exactly, the answer. **Self-RAG** style systems add a second check
on the *output*: is every claim in the answer supported by the documents? If not, generate again.

In [ ]:
class Check(BaseModel):
    """Whether an answer is supported by the documents it was based on."""
    grounded: bool = Field(description="True if every factual claim in the answer is stated in the documents.")


checker = model.with_structured_output(Check)
MAX_ATTEMPTS = 2


def check(state: RAGState) -> RAGState:
    verdict = checker.invoke(f"Documents:\n{format_docs(state['documents'])}\n\nAnswer:\n{state['answer']}")
    return {"grounded": verdict.grounded, "attempts": state["attempts"] + 1}


def after_check(state: RAGState) -> str:
    if state["grounded"] or state["attempts"] >= MAX_ATTEMPTS:
        return "done"
    return "retry"


self_rag = build_rag_graph(check_answers=True)

for label in ["distractor", "multi-hop"]:
    print(f"===== {label}: {QUESTIONS[label]}")
    final = run_graph(self_rag, QUESTIONS[label])
    print("  grounded:", final.get("grounded"), "after", final["attempts"], "check(s)")
    print()

### Code walkthrough: a retry must change something

<img src="images/agentic_rag_cards/04c_retry.png" alt="Change the prompt before retrying card: a flagged draft is not simply retried; a note saying &#x27;use only source facts&#x27; is added to the prompt, and the new draft passes. Give the retry new instructions." width="650">

**`generate` reads `grounded`.** With `temperature=0`, generating again from the same prompt gives the same answer, so a
plain retry is a wasted call. When the check failed, `generate` appends a correction to the question. A retry loop
only makes sense if something in the input is different on the next pass.

**`MAX_ATTEMPTS`** caps the loop. An answer that is still flagged after two attempts is returned with
`grounded=False` in the state, which an application can show as a warning instead of hiding.

**Look at the multi-hop result.** The answer is probably grounded (it only says what one document says) and still
incomplete. A groundedness check catches invented facts; it does not catch missing ones. Those are different
failures, and they need different fixes.

**When it is worth it.** The check is one more call per answer. It pays off where an unsupported answer is expensive
(medical, legal, grading rules) and is overkill for a chit-chat bot.

## 5. Multi-hop: the graph becomes a tool

<img src="images/agentic_rag_cards/05_multihop.png" alt="Multi-hop card: one robot finds &#x27;Sara Karimi&#x27; in the KV cache session drawer and hands the name to a second robot, who uses it at the office hours lookup to open the door to room 305." width="650">

Section 3's agent handled multi-hop questions because it could search again with what it had learned. Section 4's
graph handled routing, grading and rewriting reliably but searches once per question. Combine them with tutorial 4's
supervisor pattern: wrap the graph in a `@tool` and give it to an agent whose only job is to **break the question
into simple questions**.

```text
planner ──"Who led the LLM-501 KV cache session?"────→ self_rag graph → "Sara Karimi [llm-schedule]"
   │    ──"When and where are Sara Karimi's office hours?"→ self_rag graph → "Tuesdays 14:00 to 16:00, room 305"
   └──→ combined answer
```

In [ ]:
@tool
def ask_handbook(question: str) -> str:
    """Answer ONE simple question from the course handbook, for example 'Who leads the week 9 LLM-501
    session?'. The tool sees nothing but this question, so spell out every name, course and week."""
    return run_graph(self_rag, question, verbose=False)["answer"]


planner = create_agent(
    model,
    tools=[ask_handbook],
    system_prompt=(
        "You help students with questions about two courses. Break each question into simple questions "
        "that one handbook lookup can answer, and ask them one at a time with ask_handbook. Use earlier "
        "answers to write later questions. Then combine the answers, keeping their [source] tags. "
        "Never add facts that the tool did not return. Reply to greetings directly."
    ),
    name="planner",
)

for question in [QUESTIONS["multi-hop"],
                 "Who do I ask for more GPU hours, and when are their office hours?"]:
    print(f"===== {question}")
    show_trace(ask(planner, question)["messages"])
    print()

### Code walkthrough: who decides what

<img src="images/agentic_rag_cards/05a_responsibility.png" alt="The planner and the graph card: upstairs, the planner robot writes the next question from the clues so far; downstairs, the graph robot checks it against the handbook and sends back the result." width="650">

| Layer | Decides | Built with |
|---|---|---|
| Planner agent | Which simple questions to ask, in what order | `create_agent`, free to choose |
| `self_rag` graph | Whether to retrieve, which documents to keep, when to rewrite, whether the answer holds | Your edges, fixed |

The model has freedom where freedom helps (decomposing a question is hard to write as rules) and none where you want
guarantees (every answer the planner receives has been graded and checked).

**The docstring matters again.** "The tool sees nothing but this question" is tutorial 4's lesson about context
isolation: the inner graph has no idea what "their" or "that session" refers to, so the planner must write
self-contained questions.

**Cost.** Each tool call is a full `self_rag` run, roughly 4 to 6 model calls, plus the planner's own calls. A
two-hop question can easily cost 12 or more calls. That is the price of combining reliability with flexibility, and
it is why you route simple questions away from this path.

## 6. Measuring it, not eyeballing it

<img src="images/agentic_rag_cards/06_measure.png" alt="Same quiz, two scores card: four robots take the same pop quiz, and the scoreboard has two columns, answers passed and tokens used, with a robot weighing a pile of coins." width="650">

Every comparison so far was one question at a time, read by eye. Before choosing a design, run all of them on the
same small test set and count two things: **how many answers were right**, and **how many tokens it cost**.

The test is deliberately crude: an answer passes if it contains every expected string. Crude checks are still far
better than none, and they catch regressions every time you change a prompt.

In [ ]:
import time
from langchain_core.callbacks import get_usage_metadata_callback

EVAL = [
    ("What percentage of the LLM-501 grade is the final project?",                    ["40"]),
    ("How much do I lose per day if I hand in my LLM project late?",                   ["10"]),
    ("Which library do we have to use for the big assignment?",                        ["langgraph"]),
    ("I missed the KV cache lecture. Where can I find the person who gave it?",        ["305"]),
    ("Whose permission do I need for LLM-501 if I have not taken NLP-402?",            ["rahimi"]),
    ("I need more compute for my project. Where do I find the person in charge?",      ["moradi", "112"]),
]

SYSTEMS = {
    "naive chain":      lambda q: naive_rag(q)[0],
    "agent + tool":     lambda q: ask(rag_agent, q)["messages"][-1].content,
    "corrective graph": lambda q: run_graph(crag, q, verbose=False)["answer"],
    "planner + graph":  lambda q: ask(planner, q)["messages"][-1].content,
}

scores = {}
for name, system in SYSTEMS.items():
    passed, tokens = [], 0
    for question, expected in EVAL:
        with get_usage_metadata_callback() as usage:
            answer = system(question).lower()
        tokens += sum(u["total_tokens"] for u in usage.usage_metadata.values())
        passed.append(all(e in answer for e in expected))
        time.sleep(1)                      # be gentle with free-tier rate limits
    scores[name] = (passed, tokens)

print(f"{'system':<18}" + "".join(f"  Q{i + 1}" for i in range(len(EVAL))) + "   score   tokens")
for name, (passed, tokens) in scores.items():
    marks = "".join(f"  {'ok' if p else '--':>2}" for p in passed)
    print(f"{name:<18}{marks}   {sum(passed)}/{len(EVAL)}   {tokens:>7,}")

### Code walkthrough: reading the scoreboard

<img src="images/agentic_rag_cards/06a_scoreboard.png" alt="Count the nested calls card: tokens pour out of nested boxes labelled planner, grade and check into a total tokens jar, while another robot checks the returned answer for the expected words." width="650">

**`get_usage_metadata_callback()`** collects the token counts that Groq reports for every model call made inside the
`with` block, including calls made deep inside graphs, tools and agents. Tokens are what you pay for, so they are a
better cost measure than the number of calls.

**What you will usually see.** The naive chain is cheapest and fails Q3 to Q6. The single agent passes more, but its
score varies between runs because its strategy does. The corrective graph fixes Q2 and Q3 and still fails the
multi-hop questions. The planner passes most and costs the most, often several times the naive chain.

**Six questions is not an evaluation.** It is a smoke test. Real systems need dozens to hundreds of questions,
including questions the corpus does *not* answer, and a better check than substring matching (an LLM judge, or
checking the cited sources). The habit is what matters: a fixed test set, run after every change, with cost next to
accuracy.

The substring test also cannot score "not covered" questions, because a correct "I don't know" has no fixed wording.
Exercise 6.1 adds them.

### Exercise 6.1

<img src="images/agentic_rag_cards/06b_exercise.png" alt="Stress-test the scoreboard card: robots try an uncovered question about parking, more chunks and a larger model, and record the results. Change one thing, measure again." width="650">

1. Add two questions the handbook cannot answer ("Is there free parking?", "Who teaches CV-310?"). Write a check that
   passes if the answer admits the handbook does not cover it. Which system makes something up most often?
2. Change `retriever` to `k=4` (rebuild it with `store.as_retriever(search_kwargs={"k": 4})`) and re-run the whole
   cell. Which scores change, and what happens to the token counts of the corrective graph? Why does `k` cost more
   there than in the naive chain?
3. Switch `model` to `openai/gpt-oss-120b` and re-run. Is a bigger model a cheaper fix than a bigger graph?

## 7. Choosing between them

<img src="images/agentic_rag_cards/07_choose.png" alt="Pick the right RAG tool card: a chain for a simple lookup, a graph for questions that need decisions, and a planner for dependent clues." width="650">

| Situation | Reach for |
|---|---|
| Simple lookups, every question needs the corpus | **Chain** (tutorial 3). Cheapest, easiest to debug |
| Mix of chit-chat and lookups | Chain with a **router** in front |
| Similar documents that confuse retrieval | **Grade** retrieved documents before generating |
| Users phrase things very differently from the documents | **Rewrite** the query on a miss |
| Wrong answers are costly | **Check** the answer against the documents |
| Questions that need facts from several documents | **Planner agent** with retrieval (or the graph) as a tool |
| Long research tasks over a large corpus | A **deep agent** (tutorial 5) with the search tool, writing notes to files |

Two rules carry over from tutorials 4 and 5:

1. **Add a decision only when a test question fails without it.** Every node in section 4 costs a model call on every
   question, including the ones that did not need it.
2. **Keep the guarantees in code and the flexibility in the model.** Limits, grading and the "I don't know" path
   belong in edges. Breaking a question into parts belongs to the model.

Section 5's multi-hop fix has a weakness: it rediscovers the connection "KV cache session → Sara Karimi → room 305"
with several searches every time someone asks. If questions like that are common, it is cheaper to extract the
connections once and store them as a **knowledge graph**. That is tutorial 7.

## Glossary: quick reference

<img src="images/agentic_rag_cards/08_glossary.png" alt="The robot&#x27;s retrieval toolkit card: a toolbox holding route, retrieve, grade, rewrite, check and multi-hop." width="650">

| Term | Meaning |
|---|---|
| **Agentic RAG** | RAG where retrieval decisions (whether, what, how often, good enough?) are made at runtime by a model or a graph. |
| **Retriever as a tool** | A search function wrapped with `@tool`, so an agent decides when to call it and with what query. |
| **Routing** | Deciding whether a question needs retrieval at all, or which source to search. |
| **Document grading** | A model judging each retrieved document as relevant or not before generation. |
| **Query rewriting** | Rewording a failed search query into the vocabulary of the documents. |
| **Corrective RAG (CRAG)** | Retrieve, grade, and correct (rewrite, re-search or give up) before generating. |
| **Self-RAG** | Checking the generated answer against the documents, and regenerating if it is not supported. |
| **Grounded** | Every claim in the answer is stated in the retrieved documents. |
| **Multi-hop question** | A question whose answer needs a fact found only by using another fact first. |
| **Query decomposition** | Splitting a complex question into simple ones that one retrieval can answer each. |
| **`with_structured_output`** | Makes a model return a validated Pydantic object instead of free text. |
| **`stream_mode="updates"`** | Streams each node's name and returned keys as the graph runs. |
| **`get_usage_metadata_callback`** | Context manager that totals token usage for every model call inside it. |

---